# SEED BFCL OPD-only: train-128/validation-72, batch 64

Five epochs with two 64-task batches per epoch, one strict-Adam model update and one resumable LoRA checkpoint per batch, constant SEED-style `1e-6` learning rate with no warmup, and validation at initialization plus every epoch boundary. The first batch is a Drive-backed launch gate; the same run then resumes from checkpoint 1 through checkpoint 10.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import torch
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
assert len(gpu_names) == 1 and 'A100' in gpu_names[0].upper(), (torch.cuda.device_count(), gpu_names)
print('SEED_BFCL_A100_OK', gpu_names[0])

In [ ]:
import pathlib, subprocess, sys
SEED_URL = 'https://github.com/Alexishiyu/SEED.git'
SEED_BRANCH = 'codex/bfcl-opsd-128x72-b64'
SEED_ROOT = pathlib.Path('/content/SEED')
GORILLA_ROOT = pathlib.Path('/content/gorilla')
BFCL_ROOT = GORILLA_ROOT / 'berkeley-function-call-leaderboard'
BFCL_COMMIT = 'f7cf7359b7ac615a0b294831c5ba2bc95ee4a000'
def run(cmd, cwd=None):
    print('RUN', ' '.join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)
if not (SEED_ROOT / '.git').is_dir(): run(['git', 'clone', SEED_URL, SEED_ROOT])
run(['git', 'fetch', 'origin', SEED_BRANCH], cwd=SEED_ROOT)
seed_commit = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=SEED_ROOT, text=True).strip()
run(['git', 'checkout', '--detach', seed_commit], cwd=SEED_ROOT)
if not (GORILLA_ROOT / '.git').is_dir(): run(['git', 'clone', 'https://github.com/ShishirPatil/gorilla.git', GORILLA_ROOT])
run(['git', 'fetch', 'origin', BFCL_COMMIT], cwd=GORILLA_ROOT)
run(['git', 'checkout', '--detach', BFCL_COMMIT], cwd=GORILLA_ROOT)
run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm==0.11.0', 'peft==0.17.1', 'pandas', 'pyarrow'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/flash_attn-2.8.3%2Bcu12torch2.8cxx11abiTRUE-cp312-cp312-linux_x86_64.whl'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(SEED_ROOT)])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(BFCL_ROOT)])
run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'vllm==0.11.0'])
print('SEED_BFCL_SETUP_OK seed_sha=' + seed_commit + ' bfcl_sha=' + BFCL_COMMIT)

In [ ]:
test_files = [
    'tests/trainer/ppo/test_opd_loss.py', 'tests/trainer/ppo/test_seed_advantage.py',
    'tests/trainer/ppo/test_seed_analyzer.py', 'tests/trainer/ppo/test_episode_skill_guidance.py',
    'tests/trainer/ppo/test_seed_skill_gen_reward.py', 'tests/seed/test_june24_skill_summary.py',
    'tests/seed/test_june24_all200.py', 'tests/seed/test_june24_teacher_snapshot.py',
    'tests/trainer/ppo/test_bfcl_two_stage_optimizer.py',
    'tests/trainer/ppo/test_bfcl_checkpoint_supervisor.py',
    'tests/environments/test_bfcl_env.py', 'tests/trainer/ppo/test_opd_only_objective.py',
]
run([sys.executable, '-m', 'pytest', '-q', *test_files], cwd=SEED_ROOT)
print('SEED_BFCL_TESTS_OK')

In [ ]:
from datetime import datetime, timezone
import json
stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
RUN_ROOT = pathlib.Path('/content/drive/MyDrive/bfcl_qwen_experiment/seed_opsd_colab') / f'june24_all200_train128_val72_b64_{stamp}'
INPUTS = RUN_ROOT / 'inputs'
INPUTS.mkdir(parents=True, exist_ok=False)
SOURCE_DIRS = [
    pathlib.Path('/content/drive/MyDrive/bfcl_qwen_experiment/a100_skill_sd_50_20260624_055240/skills_openai'),
    pathlib.Path('/content/drive/MyDrive/bfcl_qwen_experiment/a100_skill_sd_150_50_199_20260624_063820/skills_openai'),
]
for source in SOURCE_DIRS: assert source.is_dir(), source
PAIRWISE_TASKS = pathlib.Path('/content/drive/MyDrive/bfcl_qwen_pairwise_analysis/15fRBFq4gbXgJeQ5CHO_bVjeEp0mlH9rB/pairwise_tasks.csv')
assert PAIRWISE_TASKS.is_file(), PAIRWISE_TASKS
cohort_manifest = INPUTS / 'june24_all200_cohort.json'
split_manifest = INPUTS / 'june24_train128_val72_b64_split.json'
skill_bank = INPUTS / 'june24_train128_skill_bank.json'
builder = SEED_ROOT / 'scripts/build_june24_all200_skill_bank.py'
build_cmd = [sys.executable, str(builder), '--pairwise-tasks-csv', str(PAIRWISE_TASKS), '--cohort-manifest', str(cohort_manifest), '--split-manifest', str(split_manifest), '--output', str(skill_bank), '--split-profile', '128x72_b64', '--batch-size', '64']
for source in SOURCE_DIRS: build_cmd += ['--source-dir', str(source)]
for task_id in ['multi_turn_base_56', 'multi_turn_base_154', 'multi_turn_base_169']: build_cmd += ['--allow-repaired-task-id', task_id]
run(build_cmd, cwd=SEED_ROOT)
split_payload = json.loads(split_manifest.read_text(encoding='utf-8'))
assert split_payload['train']['classification_counts'] == {'fixed': 26, 'both_wrong': 72, 'harmed': 8, 'both_correct': 22}
assert split_payload['validation']['classification_counts'] == {'fixed': 14, 'both_wrong': 40, 'harmed': 5, 'both_correct': 13}
(RUN_ROOT / 'metadata').mkdir(parents=True, exist_ok=True)
(RUN_ROOT / 'metadata' / 'resolved_seed_commit.txt').write_text(seed_commit + '\n', encoding='utf-8')
print('SEED_BFCL_128_72_INPUTS_OK run_root=' + str(RUN_ROOT))

In [ ]:
MODEL = 'Qwen/Qwen3-4B-Instruct-2507'
RLPAPER_SHA = '690946ca3e1d2f257d7c1acdff4df4eb1feed1ec'
launcher = SEED_ROOT / 'examples/seed_trainer/run_bfcl_opsd.py'
common = [
    sys.executable, str(launcher), '--june24-skill-bank', str(skill_bank),
    '--cohort-manifest', str(cohort_manifest), '--split-manifest', str(split_manifest),
    '--run-root', str(RUN_ROOT), '--bfcl-root', str(BFCL_ROOT), '--model', MODEL,
    '--iterations', '5', '--batch-size', '64', '--checkpoint-updates', '1',
    '--optimizer', 'adam', '--lr-schedule', 'constant', '--final-lr', '1e-6',
    '--inline-same-prompt-diagnostics', '--resume', 'auto', '--rlpaper-sha', RLPAPER_SHA,
]
run(common, cwd=SEED_ROOT)
plan = json.loads((RUN_ROOT / 'metadata/privileged_june24_plan.json').read_text(encoding='utf-8'))
assert plan['provenance']['training_rollouts'] == 640
assert plan['provenance']['validation_rollouts'] == 432
assert plan['provenance']['total_updates'] == 10
assert plan['provenance']['checkpoint_schedule'] == list(range(1, 11))
assert plan['provenance']['validation_steps'] == [0, 2, 4, 6, 8, 10]
assert plan['provenance']['learning_rate'] == {'style': 'constant', 'warmup_updates': 0, 'warmup_target_lr': None, 'final_lr': 1e-6}
print('SEED_BFCL_128_72_PREFLIGHT_OK')

In [ ]:
run(common + ['--execute', '--stop-after-update', '1'], cwd=SEED_ROOT)
checkpoint_root = RUN_ROOT / 'checkpoints' / 'privileged_june24'
assert (checkpoint_root / 'latest_checkpointed_iteration.txt').read_text().strip() == '1'
assert (RUN_ROOT / 'evidence/privileged_june24/updates/step_000001.json').is_file()
assert list((checkpoint_root / 'global_step_1/actor').glob('optim_world_size_*_rank_*.pt'))
assert (checkpoint_root / 'global_step_1/actor/lora_adapter/adapter_model.safetensors').is_file()
print('SEED_BFCL_BATCH64_DRIVE_SMOKE_OK checkpoint=1')

In [ ]:
run(common + ['--execute'], cwd=SEED_ROOT)
assert (checkpoint_root / 'latest_checkpointed_iteration.txt').read_text().strip() == '10'
print('SEED_BFCL_128_72_FIVE_EPOCH_TRAIN_OK')

In [ ]:
export_root = RUN_ROOT / 'exports' / 'privileged_june24_merged'
export_evidence = RUN_ROOT / 'evidence' / 'privileged_june24' / 'checkpoint_export.json'
run([sys.executable, str(SEED_ROOT / 'examples/seed_trainer/merge_bfcl_opsd_lora.py'), '--checkpoint-root', str(checkpoint_root), '--base-model', MODEL, '--output', str(export_root), '--evidence', str(export_evidence), '--validate-vllm'], cwd=SEED_ROOT)
print('SEED_BFCL_FINAL_MERGE_RELOAD_OK')

In [ ]:
final_report = RUN_ROOT / 'evidence' / 'seed_bfcl_five_pass_complete.json'
run([sys.executable, str(SEED_ROOT / 'examples/seed_trainer/collect_bfcl_opsd_five_pass_evidence.py'), '--run-root', str(RUN_ROOT), '--export-evidence', str(export_evidence), '--output', str(final_report)], cwd=SEED_ROOT)
report = json.loads(final_report.read_text(encoding='utf-8'))
assert report['status'] == 'complete'
print('SEED_BFCL_128_72_COMPLETE', final_report)
print(json.dumps(report['validation_curve'], indent=2))